In [1]:
# NOTEBOOK 02: Transform & Create Staging Tables (with CTEs)
# Purpose: Clean, deduplicate, and extract skills from raw data
# ==============================================================

# Step 1: Create a temporary view for CTEs
spark.sql("""
-- CTE 1: Deduplicate and clean raw data
WITH raw_deduplicated AS (
    SELECT 
        job_id,
        TRIM(job_title) AS job_title,
        TRIM(company_name) AS company_name,
        TRIM(location) AS location,
        salary_min,
        salary_max,
        LOWER(job_description) AS job_description,
        posted_date,
        ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY posted_date DESC) AS rn
    FROM stg_job_postings_raw
    WHERE job_id IS NOT NULL 
        AND job_title IS NOT NULL 
        AND company_name IS NOT NULL
),

-- CTE 2: Remove duplicates (keep latest)
cleaned_jobs AS (
    SELECT 
        job_id,
        job_title,
        company_name,
        location,
        salary_min,
        salary_max,
        job_description,
        posted_date,
        CURRENT_TIMESTAMP() AS ingestion_date
    FROM raw_deduplicated
    WHERE rn = 1
),

-- CTE 3: Extract skills from job description
skills_extracted AS (
    SELECT 
        job_id,
        CASE 
            WHEN job_description LIKE '%sql%' THEN 'SQL'
            WHEN job_description LIKE '%python%' THEN 'Python'
            WHEN job_description LIKE '%power bi%' THEN 'Power BI'
            WHEN job_description LIKE '%tableau%' THEN 'Tableau'
            WHEN job_description LIKE '%spark%' THEN 'Spark'
            WHEN job_description LIKE '%azure%' THEN 'Azure'
            WHEN job_description LIKE '%aws%' THEN 'AWS'
            WHEN job_description LIKE '%x12%' THEN 'X12 EDI'
            WHEN job_description LIKE '%edi%' THEN 'EDI'
            WHEN job_description LIKE '%machine learning%' THEN 'Machine Learning'
            WHEN job_description LIKE '%tensorflow%' THEN 'TensorFlow'
            WHEN job_description LIKE '%excel%' THEN 'Excel'
            WHEN job_description LIKE '%redshift%' THEN 'Redshift'
            WHEN job_description LIKE '%bigquery%' THEN 'BigQuery'
            WHEN job_description LIKE '%dbt%' THEN 'dbt'
            WHEN job_description LIKE '%airflow%' THEN 'Airflow'
            WHEN job_description LIKE '%hadoop%' THEN 'Hadoop'
        END AS skill_found
    FROM cleaned_jobs
    WHERE job_description LIKE '%sql%' 
        OR job_description LIKE '%python%' 
        OR job_description LIKE '%power bi%'
        OR job_description LIKE '%tableau%'
        OR job_description LIKE '%spark%'
        OR job_description LIKE '%azure%'
        OR job_description LIKE '%aws%'
        OR job_description LIKE '%x12%'
        OR job_description LIKE '%edi%'
        OR job_description LIKE '%machine learning%'
        OR job_description LIKE '%tensorflow%'
        OR job_description LIKE '%excel%'
        OR job_description LIKE '%redshift%'
        OR job_description LIKE '%bigquery%'
        OR job_description LIKE '%dbt%'
        OR job_description LIKE '%airflow%'
        OR job_description LIKE '%hadoop%'
)

-- Final Output: Create stg_jobs_cleaned table
SELECT * FROM cleaned_jobs
""")

# Save as staging table
spark.sql("""
    CREATE OR REPLACE TABLE stg_jobs_cleaned AS
    WITH raw_deduplicated AS (
        SELECT 
            job_id,
            TRIM(job_title) AS job_title,
            TRIM(company_name) AS company_name,
            TRIM(location) AS location,
            salary_min,
            salary_max,
            LOWER(job_description) AS job_description,
            posted_date,
            ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY posted_date DESC) AS rn
        FROM stg_job_postings_raw
        WHERE job_id IS NOT NULL 
            AND job_title IS NOT NULL 
            AND company_name IS NOT NULL
    )
    SELECT 
        job_id,
        job_title,
        company_name,
        location,
        salary_min,
        salary_max,
        job_description,
        posted_date,
        CURRENT_TIMESTAMP() AS ingestion_date
    FROM raw_deduplicated
    WHERE rn = 1
""")

print("✅ Table 'stg_jobs_cleaned' created!")

# Step 2: Create stg_companies_cleaned (distinct companies)
spark.sql("""
    CREATE OR REPLACE TABLE stg_companies_cleaned AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY company_name) AS company_id,
        company_name,
        'Unknown' AS industry,
        CURRENT_TIMESTAMP() AS created_date
    FROM (
        SELECT DISTINCT company_name
        FROM stg_jobs_cleaned
    )
""")

print("✅ Table 'stg_companies_cleaned' created!")

# Step 3: Create stg_skills_extracted (all unique skills)
spark.sql("""
    CREATE OR REPLACE TABLE stg_skills_extracted AS
    SELECT 
        ROW_NUMBER() OVER (ORDER BY skill) AS skill_id,
        skill,
        'Technical' AS skill_category,
        CURRENT_TIMESTAMP() AS created_date
    FROM (
        SELECT DISTINCT 
            CASE 
                WHEN job_description LIKE '%sql%' THEN 'SQL'
                WHEN job_description LIKE '%python%' THEN 'Python'
                WHEN job_description LIKE '%power bi%' THEN 'Power BI'
                WHEN job_description LIKE '%tableau%' THEN 'Tableau'
                WHEN job_description LIKE '%spark%' THEN 'Spark'
                WHEN job_description LIKE '%azure%' THEN 'Azure'
                WHEN job_description LIKE '%aws%' THEN 'AWS'
                WHEN job_description LIKE '%x12%' THEN 'X12 EDI'
                WHEN job_description LIKE '%edi%' THEN 'EDI'
                WHEN job_description LIKE '%machine learning%' THEN 'Machine Learning'
                WHEN job_description LIKE '%tensorflow%' THEN 'TensorFlow'
                WHEN job_description LIKE '%excel%' THEN 'Excel'
                WHEN job_description LIKE '%redshift%' THEN 'Redshift'
                WHEN job_description LIKE '%bigquery%' THEN 'BigQuery'
                WHEN job_description LIKE '%dbt%' THEN 'dbt'
            END AS skill
        FROM stg_jobs_cleaned
        WHERE job_description LIKE '%sql%' 
            OR job_description LIKE '%python%' 
            OR job_description LIKE '%power bi%'
            OR job_description LIKE '%tableau%'
            OR job_description LIKE '%spark%'
            OR job_description LIKE '%azure%'
            OR job_description LIKE '%aws%'
            OR job_description LIKE '%x12%'
            OR job_description LIKE '%edi%'
            OR job_description LIKE '%machine learning%'
            OR job_description LIKE '%tensorflow%'
            OR job_description LIKE '%excel%'
            OR job_description LIKE '%redshift%'
            OR job_description LIKE '%bigquery%'
            OR job_description LIKE '%dbt%'
    )
""")

print("✅ Table 'stg_skills_extracted' created!")

# Step 4: Show record counts
print("\n📊 STAGING TABLES CREATED:")
spark.sql("SELECT 'stg_jobs_cleaned' as table_name, COUNT(*) as record_count FROM stg_jobs_cleaned").show()
spark.sql("SELECT 'stg_companies_cleaned' as table_name, COUNT(*) as record_count FROM stg_companies_cleaned").show()
spark.sql("SELECT 'stg_skills_extracted' as table_name, COUNT(*) as record_count FROM stg_skills_extracted").show()

print("\n✅ Phase 2 Complete: All Staging Tables Created!")

StatementMeta(, 68d5f7c8-0c0f-4ea5-8b1d-8f7c2655789e, 3, Finished, Available, Finished, False)

✅ Table 'stg_jobs_cleaned' created!
✅ Table 'stg_companies_cleaned' created!
✅ Table 'stg_skills_extracted' created!

📊 STAGING TABLES CREATED:
+----------------+------------+
|      table_name|record_count|
+----------------+------------+
|stg_jobs_cleaned|          50|
+----------------+------------+

+--------------------+------------+
|          table_name|record_count|
+--------------------+------------+
|stg_companies_cle...|          47|
+--------------------+------------+

+--------------------+------------+
|          table_name|record_count|
+--------------------+------------+
|stg_skills_extracted|           5|
+--------------------+------------+


✅ Phase 2 Complete: All Staging Tables Created!
